In [9]:
import numpy as np
import math
from scipy.optimize import minimize

In [10]:
def prelec_weight(p, alpha):
    return np.exp(-((-np.log(p)) ** alpha))

def cpt_value(q, alpha, beta, lam, x0=0):
    diff = q - x0
    if diff >= 0:
        return diff ** alpha
    return -lam * ((-diff) ** beta)

def cpt_subjective_value(p, q_pref, q_no, prelec_alpha, value_alpha, value_beta, lam):
    w = prelec_weight(p, prelec_alpha)
    u_pref = cpt_value(q_pref, value_alpha, value_beta, lam)
    u_no = cpt_value(q_no, value_alpha, value_beta, lam)
    return w * (u_pref - u_no)

In [ ]:
#### Logistic choice probability
def logistic(x):
    return 1 / (1 + np.exp(-x))

#### Negative log-likelihood
def neg_log_likelihood(params, p, q_pref, q_no, C):
    prelec_alpha, alpha, beta, lam, theta = params
    
    V = np.array([
        cpt_subjective_value(
            p[i], 
            q_pref[i], 
            q_no[i],
            prelec_alpha, 
            alpha, 
            beta, 
            lam
        ) for i in range(len(p))
    ])
    
    probs = logistic(theta * V)
    eps = 1e-9
    ll = np.sum(C * np.log(probs + eps) + (1 - C) * np.log(1 - probs + eps))
    return -ll  # minimize negative log lik

In [ ]:
from importnb import Notebook
with Notebook():
    from Sources.Examples.LabSimpleEnv_2 import simulate_viewport_with_tiles, CPTPrefetchEnvSimple
    from Sources.Examples.LabSimpleEnv_2 import zipf, n_frames, n, num_tiles

window_len = 10
n_users = 30
total_videos = 100
capacity = 50
step_size = 1.5

yaw, pitch, tiles_per_frame, tiles_array = simulate_viewport_with_tiles(
    num_steps=n_frames,
    n=n,
    fov_yaw=120,
    fov_pitch=60,
    damping=0.99,
    step_size=step_size,
    start_yaw=180,
    start_pitch=0
)

user_tiles = [
    {
        "tiles": tiles_per_frame[start:(start + window_len)],
        "yaw": yaw[start:(start + window_len)],
        "pitch": pitch[start:(start + window_len)],
    } for start in range(0, n_users * window_len, window_len)
]

videos_requests_idx = zipf(n_users, total_videos=total_videos, alpha=1.0)

env = CPTPrefetchEnvSimple(
    num_tiles=num_tiles,
    n_users=n_users,
    cache_capacity=capacity,
    user_tiles=user_tiles,
    videos_requests=videos_requests_idx
)

obs = env.reset()
max_steps = window_len

p_data, q_pref, q_no, C_data = [], [], [], []



In [ ]:
#### Example synthetic data
n = 50
p_data = np.random.uniform(0.05, 0.9, n)
q_pref = np.random.uniform(50, 90, n)
q_no = q_pref - np.random.uniform(5, 25, n)

#### synthetic choices: pretend user behaves like CPT(0.7,0.9,0.9,2.0)
true_params = (0.7, 0.9, 0.9, 2.0, 0.1)
V_true = np.array([
    cpt_subjective_value(
        p_data[i], 
        q_pref[i], 
        q_no[i],
        true_params[0], 
        true_params[1], 
        true_params[2], 
        true_params[3]
    ) for i in range(n)
])
C_data = (np.random.rand(n) < logistic(true_params[-1] * V_true)).astype(int)

#### Fit parameters
x0 = np.array([0.6, 0.8, 0.8, 1.5, 0.1])  # initial guess
bounds = [(0.01, 1.5), (0.1, 2), (0.1, 2), (0.1, 5), (0.01, 10)]

res = minimize(
    neg_log_likelihood, 
    x0,
    args=(p_data, q_pref, q_no, C_data),
    bounds=bounds
)

print("Estimated parameters:")
print("Prelec alpha =", res.x[0])
print("Value alpha  =", res.x[1])
print("Value beta   =", res.x[2])
print("Lambda       =", res.x[3])
print("Theta        =", res.x[4])

Estimated parameters:
Prelec alpha = 0.01
Value alpha  = 0.30024726768787774
Value beta   = 0.8
Lambda       = 1.5
Theta        = 10.0
